### Chapter 2.4 - Calculus

Calculus studies how quantities change. Machine learning uses calculus to answer one central question: if we change a model parameter a little, how does the loss change?

This chapter focuses on derivatives, gradients, and the chain rule as practical tools for learning systems.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt


### 2.4.1 Derivatives and Differentiation

#### 1. Intuition

A derivative measures the slope of a function at a point. A function is a rule that maps an input to an output.

For example, `f(x) = x * x` maps `2` to `4` and `3` to `9`. The derivative tells us how fast the output changes when the input changes a tiny amount.

#### 2. Why this exists

Learning requires knowing which direction changes the loss. If increasing a parameter increases the loss, we may want to decrease that parameter. If increasing it decreases the loss, we may want to increase it.


#### 3. Examples

Estimate a derivative manually using a small change. This is called a finite difference.


In [ ]:
def f(x):
    return x * x

x = 3.0
h = 0.001
slope = (f(x + h) - f(x)) / h
slope


Compare with the exact derivative of `x * x`, which is `2 * x`.


In [ ]:
x = 3.0
exact = 2 * x
exact


PyTorch can compute this derivative automatically when requested.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x * x
y.backward()
x.grad


#### 4. Step-by-step breakdown

`def f(x):` defines a Python function named `f`.

`return x * x` says the function output is the square of the input.

`h` is a small positive number.

`f(x + h) - f(x)` measures how much the output changed.

Dividing by `h` turns that output change into a change-per-unit-input estimate.

`requires_grad=True` tells PyTorch to track operations involving `x` so it can compute derivatives later.

`y.backward()` starts derivative computation from `y`.

`x.grad` stores the derivative of `y` with respect to `x`.

#### 5. Connection to ML systems

Gradient-based learning repeatedly uses derivatives to decide how to adjust parameters. A parameter is a model value learned from data, such as a weight or bias.

#### 6. Common confusion points

- A derivative is local. It describes change near one point.
- A small finite difference is an approximation, not the exact derivative.
- `requires_grad=True` does not compute a derivative immediately. It records operations so derivatives can be computed later.
- `.backward()` is the request to compute gradients.


### 2.4.2 Visualization Utilities

#### 1. Intuition

Visualization means turning numbers into a picture. For calculus, plotting a function helps connect formulas to shape.

An axis is a number line in a plot. The horizontal axis often shows input `x`. The vertical axis often shows output `f(x)`.

#### 2. Why this exists

Plots make slopes easier to see. A steep curve means small input changes create large output changes. A flat curve means small input changes create small output changes.


#### 3. Examples

Create points for a simple curve. Sampling means choosing input values where we evaluate the function.


In [ ]:
def f(x):
    return x * x

x = np.linspace(-2, 2, 5)
y = f(x)
x, y


Plot the curve. This cell produces a figure when executed in a notebook.


In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(x, y, marker="o")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid(True)


#### 4. Step-by-step breakdown

`np.linspace(-2, 2, 5)` creates 5 evenly spaced values from -2 to 2.

`y = f(x)` applies the function to every NumPy input value.

`plt.figure(figsize=(4, 3))` creates a small plotting area.

`plt.plot(x, y, marker="o")` draws the curve through the sampled points.

`xlabel`, `ylabel`, and `grid` make the plot easier to read.

#### 5. Connection to ML systems

Training curves plot loss over time. They help reveal whether learning is improving, stuck, unstable, or overfitting.

Overfitting means the model performs well on training data but poorly on new data.

#### 6. Common confusion points

- A plot is not the function itself; it is sampled evidence about the function.
- More sample points usually make a curve look smoother.
- Axes labels matter because they explain what the numbers mean.
- A visual trend still needs numerical confirmation.


### 2.4.3 Partial Derivatives and Gradients

#### 1. Intuition

A partial derivative measures how a function changes with respect to one input while holding the other inputs fixed.

A gradient is a vector of partial derivatives. It points in the direction where the function increases fastest locally.

#### 2. Why this exists

Models usually have many parameters. We need to know how the loss changes with respect to each parameter, not just one number.


#### 3. Examples

Manual finite difference for a function with two inputs.


In [ ]:
def f(x, y):
    return x * x + 3 * y

x, y, h = 2.0, 1.0, 0.001
dx = (f(x + h, y) - f(x, y)) / h
dy = (f(x, y + h) - f(x, y)) / h
dx, dy


PyTorch gradient for the same function.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(1.0, requires_grad=True)
z = x * x + 3 * y
z.backward()
x.grad, y.grad


#### 4. Step-by-step breakdown

`dx` estimates the partial derivative with respect to `x`. Only `x` is changed by `h`.

`dy` estimates the partial derivative with respect to `y`. Only `y` is changed by `h`.

In the PyTorch version, both `x` and `y` are marked with `requires_grad=True`.

After `z.backward()`, `x.grad` stores the partial derivative with respect to `x`, and `y.grad` stores the partial derivative with respect to `y`.

#### 5. Connection to ML systems

A model with thousands or billions of parameters has a huge gradient. Conceptually, it is still the same idea: one slope per parameter.

#### 6. Common confusion points

- A partial derivative changes one input at a time.
- A gradient collects partial derivatives into one object.
- The gradient depends on the current input values.
- A gradient points toward increase; learning often moves in the opposite direction to reduce loss.


### 2.4.4 Chain Rule

#### 1. Intuition

The chain rule explains how derivatives pass through composed functions.

A composed function is a function built by feeding one function's output into another function. If `u = g(x)` and `y = f(u)`, then `y` depends on `x` through `u`.

#### 2. Why this exists

Neural networks are long chains of operations. The chain rule is the reason we can compute how early parameters affect a final loss.


#### 3. Examples

Manual chain rule for `u = 2x` and `y = u^2`.


In [ ]:
x = 3.0
u = 2 * x
du_dx = 2
dy_du = 2 * u
dy_dx = dy_du * du_dx
dy_dx


PyTorch computes the same chained derivative.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)
u = 2 * x
y = u * u
y.backward()
x.grad


#### 4. Step-by-step breakdown

`u = 2 * x` is the inner function.

`y = u * u` is the outer function.

`du_dx = 2` says `u` changes 2 units for each 1 unit change in `x`.

`dy_du = 2 * u` says `y` changes based on the current value of `u`.

The chain rule multiplies these slopes: `dy_dx = dy_du * du_dx`.

PyTorch records the operations and applies this rule automatically during `backward()`.

#### 5. Connection to ML systems

Backpropagation is the efficient application of the chain rule through a computation graph. A computation graph is a record of which values were produced from which operations.

#### 6. Common confusion points

- The chain rule multiplies local derivatives along a path.
- Intermediate values matter because derivatives are evaluated at current values.
- Backpropagation is not magic; it is organized chain rule bookkeeping.
- If a value is detached from the graph, gradients stop flowing through that path.


### 2.4.5 Discussion

#### 1. Intuition

Calculus gives machine learning a way to improve by small, directed changes.

The derivative tells us local sensitivity. The gradient extends that idea to many parameters. The chain rule connects many operations together.

#### 2. Why this exists

Without calculus, we could still try random parameter changes, but learning would be much less efficient. Derivatives tell us which changes are promising.


#### 3. Examples

A single gradient-descent-style step can be written directly.


In [ ]:
w = torch.tensor(3.0)
grad = torch.tensor(6.0)
learning_rate = 0.1
new_w = w - learning_rate * grad
new_w


#### 4. Step-by-step breakdown

`w` represents a parameter.

`grad` represents the derivative of the loss with respect to `w`.

`learning_rate` controls the step size.

`w - learning_rate * grad` moves opposite the gradient. Opposite the gradient means moving toward local decrease, because the gradient points toward local increase.

#### 5. Connection to ML systems

Optimization algorithms build on this update idea. Optimization means choosing values that make an objective better. In training, the objective is usually to reduce loss.

#### 6. Common confusion points

- A derivative is not a full explanation of the whole function.
- Gradients are local, so a good small step can still be part of a larger difficult path.
- Learning rate matters because gradients provide direction, not automatically the perfect distance.
- Calculus tells us how to change parameters; data and objectives tell us what changes are useful.


### 2.4.6 Exercises

#### 1. Intuition

These exercises practice estimating and interpreting derivatives.

#### 2. Why this exists

Derivative fluency makes automatic differentiation easier to trust and debug.


#### 3. Examples

Exercise 1: Estimate the derivative of `f(x) = x^3` at `x = 2`.


In [ ]:
def f(x):
    return x ** 3

x, h = 2.0, 0.001
estimate = (f(x + h) - f(x)) / h
estimate


Exercise 2: Use PyTorch to compute the exact derivative at the same point.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3
y.backward()
x.grad


Exercise 3: Compute partial derivatives for `z = x*y + y`.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)
z = x * y + y
z.backward()
x.grad, y.grad


#### 4. Step-by-step breakdown

Exercise 1 uses finite differences.

Exercise 2 uses automatic differentiation.

Exercise 3 checks that a function can depend on more than one variable.

#### 5. Connection to ML systems

These exercises are small versions of what happens during model training: compute a loss, call backward, read gradients, and update parameters.

#### 6. Common confusion points

- The finite-difference estimate depends on `h`.
- Gradients accumulate in PyTorch unless cleared in training loops.
- A tensor must require gradients before PyTorch tracks its operations.
- A scalar output makes `.backward()` easiest to use.
